In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [26]:
import torch
import torch.nn as nn
import math


class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2

        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor):
        # # Flatten patches and project
        # print("PatchEmbedding x shape 1 =  ",x.shape)
        x = self.projection(x)  # (batch_size, embed_dim, grid_size, grid_size)
        # print(" PatchEmbedding x shape 2 =  ",x.shape)
        x = x.flatten(2)  # (batch_size, embed_dim, num_patches)
        # print(" PatchEmbedding x shape 3 =  ",x.shape)
        x = x.transpose(1, 2)  # (batch_size, num_patches, embed_dim)
        # print(" PatchEmbedding x shape 4 =  ",x.shape)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        """ 
            scale = 1 / sqrt(dk)
        """
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.attn_dropout = nn.Dropout(0.1)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor):
        # print(" MultiHeadSelfAttention x shape 1 =  ",x.shape)
        batch_size, num_tokens, embed_dim = x.shape
        qkv = self.qkv(x)  # (batch_size, num_tokens, embed_dim * 3)
        # print(" MultiHeadSelfAttention qkv shape 1 =  ",qkv.shape)
        qkv = qkv.reshape(batch_size, num_tokens, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        # print(" MultiHeadSelfAttention qkv shape 2 =  ",qkv.shape)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Each has shape (batch_size, num_heads, num_tokens, head_dim)

            # @ = matrix multiplication
            # transpose = swapping columns
        attn_scores = (q @ k.transpose(-2, -1)) * self.scale  # (batch_size, num_heads, num_tokens, num_tokens)
        attn_weights = attn_scores.softmax(dim=-1)  # softmax on the last dimension of the tensor matrix
        attn_weights = self.attn_dropout(attn_weights)  # (batch_size, num_heads, num_tokens, num_tokens)

        # print((attn_weights @ v).transpose(1, 2).shape)
        attn_output = (attn_weights @ v).transpose(1, 2).reshape(batch_size, num_tokens, embed_dim)
        output = self.proj(attn_output)
        return output

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.self_attn(self.norm1(x))   # residual connection + normalize matrix
        x = x + self.mlp(self.norm2(x))
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, num_classes=1000, 
                 embed_dim=768, num_heads=12, depth=12, mlp_dim=3072, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)

            # ( * ) operator unpacks the list of layers, 
        self.transformer = nn.Sequential(
            *[TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)]
        )

        # self.transformer = nn.ModuleList([
        #     TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)
        # ])

        # self.transformer = nn.ModuleDict({
        #     f'layer_{i}': TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) 
        #     for i in range(depth)
        # })

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)  # (batch_size, num_patches, embed_dim)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # (batch_size, 1, embed_dim)

        x = torch.cat((cls_tokens, x), dim=1)  # (batch_size, num_patches + 1, embed_dim)
        x = x + self.pos_embed
        x = self.pos_dropout(x)

            # transformer in a sequential
        x = self.transformer(x)  # (batch_size, num_patches + 1, embed_dim)

        # for layer in self.transformer:   # transformer in a module list
        #     x = layer(x)

        # for layer in self.transformer.values():  # # transformer in a module dict
        #     x = layer(x)
        print(x.shape)
        x = self.norm(x)

        cls_output = x[:, 0]  # (batch_size, embed_dim)
        output = self.head(cls_output)  # (batch_size, num_classes)
        return output

# Example usage
if __name__ == "__main__":
    model = VisionTransformer()
    dummy_input = torch.randn(1, 3, 224, 224)
    # print(dummy_input[0])
    output = model(dummy_input)
    # print(output.shape)  # Should print (1, 1000)
    # print(output[0].argmax(dim =0))


torch.Size([1, 197, 768])


In [ ]:
# dummy_input = torch.randn(1, 3, 224, 224)
# dummy_input = dummy_input.flatten(2)
# print(dummy_input.shape)
# dummy_input = dummy_input.transpose(-2, -1)
# print(dummy_input.shape)
# # dummy_input.softmax(dim=-1)

# dum = dummy_input.expand(5, -1, -1)
# dum.shape

In [9]:
import torch
import torch.nn as nn



class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2

        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor):
        # Flatten patches and project
        # print(" PatchEmbedding x shape 1 =  ",x.shape)
        x = self.projection(x)  # (batch_size, embed_dim, grid_size, grid_size)
        # print(" PatchEmbedding x shape 2 =  ",x.shape)
        x = x.flatten(2)  # (batch_size, embed_dim, num_patches)
        # print(" PatchEmbedding x shape 3 =  ",x.shape)
        x = x.transpose(1, 2)  # (batch_size, num_patches, embed_dim)
        # print(" PatchEmbedding x shape 4 =  ",x.shape)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.attn_dropout = nn.Dropout(0.1)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor):
        # print(" MultiHeadSelfAttention x shape 1 =  ",x.shape)
        batch_size, num_tokens, embed_dim = x.shape
        qkv = self.qkv(x)  # (batch_size, num_tokens, embed_dim * 3)
        # print(" MultiHeadSelfAttention qkv shape 1 =  ",qkv.shape)
        qkv = qkv.reshape(batch_size, num_tokens, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        # print(" MultiHeadSelfAttention qkv shape 2 =  ",qkv.shape)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Each has shape (batch_size, num_heads, num_tokens, head_dim)

            # @ = matrix multiplication
            # transpose = swapping columns
        attn_scores = (q @ k.transpose(-2, -1)) * self.scale  # (batch_size, num_heads, num_tokens, num_tokens)
        attn_weights = attn_scores.softmax(dim=-1)  # softmax on the last dimension of the tensor matrix
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = (attn_weights @ v).transpose(1, 2).reshape(batch_size, num_tokens, embed_dim)
        output = self.proj(attn_output)
        return output

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.self_attn(self.norm1(x))   # residual connection + normalize matrix
        x = x + self.mlp(self.norm2(x))
        return x



class VisionTransformerForDetection(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, num_classes=91,
                 embed_dim=768, num_heads=12, depth=12, mlp_dim=3072, num_queries=100, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)

        self.transformer = nn.Sequential(
            *[TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)]
        )

        self.norm = nn.LayerNorm(embed_dim)

        # Object queries
        self.query_embed = nn.Parameter(torch.randn(num_queries, embed_dim))

        # Detection heads
        self.class_head = nn.Linear(embed_dim, num_classes)  # Class logits
        self.bbox_head = nn.Linear(embed_dim, 4)  # Bounding box coordinates

        self._init_weights()

        print("Total paramets of VisionTransformerForDetection : ",sum(p.numel() for p in self.parameters()))

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.query_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)  # (batch_size, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # (batch_size, 1, embed_dim)
        x = torch.cat((cls_tokens, x), dim=1)  # (batch_size, num_patches + 1, embed_dim)
        x = x + self.pos_embed
        x = self.pos_dropout(x)

        x = self.transformer(x)  # (batch_size, num_patches + 1, embed_dim)
        x = self.norm(x)

        # Object queries
        queries = self.query_embed.unsqueeze(0).expand(batch_size, -1, -1)  # (batch_size, num_queries, embed_dim)
        outputs = x[:, 1:]  # Skip the cls_token

        # Detection outputs
        class_logits = self.class_head(queries)  # (batch_size, num_queries, num_classes)
        bbox_preds = self.bbox_head(queries)  # (batch_size, num_queries, 4)
        return class_logits, bbox_preds

# Example usage
if __name__ == "__main__":
    model = VisionTransformerForDetection(num_classes=91, num_queries=100)
    dummy_input = torch.randn(1, 3, 224, 224)
    class_logits, bbox_preds = model(dummy_input)
    print(class_logits.shape)  # (1, 100, 91)
    print(bbox_preds.shape)    # (1, 100, 4)


Total paramets of VisionTransformerForDetection :  85920863
torch.Size([1, 100, 91])
torch.Size([1, 100, 4])


In [ ]:
for i in model.state_dict().keys():
    print(i)

print("Total parameters :- ",sum(p.numel() for n, p in model.named_parameters()))

for name, param in model.named_parameters():
    print(f"Parameter Name: {name}")
    print(f"Parameter Shape: {param.shape}")
    print(f"Parameter Dimention: {param.dim()}")
    print(f"Parameter numel or total number of parameters: {param.numel()}")
    print(f"Requires Grad: {param.requires_grad}\n")

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

# Example Dataset
class ObjectDetectionDataset(Dataset):
    def __init__(self, images, annotations):
        self.images = images  # List of images (X)
        self.annotations = annotations  # List of annotations (Y)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]  # Image (X)
        annotation = self.annotations[idx]  # Annotation (Y): {"boxes": ..., "labels": ...}
        return image, annotation


class DetectionLoss(nn.Module):
    def __init__(self, num_classes, num_queries):
        super().__init__()
        self.class_loss_fn = nn.CrossEntropyLoss()
        self.bbox_loss_fn = nn.L1Loss()
        self.num_queries = num_queries
        self.no_object_class = 0  # Reserve class 0 for "no object"

    def forward(self, pred_class_logits, pred_bboxes, target_labels, target_bboxes):
        batch_size, num_queries, num_classes = pred_class_logits.shape

        # Initialize padded labels and bounding boxes
        padded_labels = torch.full((batch_size, self.num_queries), self.no_object_class, dtype=torch.long, device=target_labels.device)
        padded_bboxes = torch.zeros((batch_size, self.num_queries, 4), dtype=torch.float, device=target_bboxes.device)

        # Mask for valid objects
        object_mask = torch.zeros((batch_size, self.num_queries), dtype=torch.bool, device=target_labels.device)

        # Fill in the targets
        for i in range(batch_size):
            num_objects = target_labels[i].shape[0]
            padded_labels[i, :num_objects] = target_labels[i]
            padded_bboxes[i, :num_objects] = target_bboxes[i]
            object_mask[i, :num_objects] = True

        # Classification Loss
        pred_class_logits = pred_class_logits.view(-1, num_classes)  # Flatten predictions
        padded_labels = padded_labels.view(-1)  # Flatten targets
        class_loss = self.class_loss_fn(pred_class_logits, padded_labels)

        # Bounding Box Loss (mask no-object predictions)
        pred_bboxes = pred_bboxes * object_mask.unsqueeze(-1)  # Apply mask to predicted boxes
        padded_bboxes = padded_bboxes * object_mask.unsqueeze(-1)  # Apply mask to target boxes
        bbox_loss = self.bbox_loss_fn(pred_bboxes, padded_bboxes)

        total_loss = class_loss + bbox_loss
        return total_loss



def train_model(model, dataloader, optimizer, device):
    model.train()
    loss_fn = DetectionLoss(num_classes=91, num_queries=100)
    
    for batch in dataloader:
        images, annotations = batch

        images = images.to(device)
        # target_labels = [a["labels"].to(device) for a in annotations]
        # target_bboxes = [a["boxes"].to(device) for a in annotations]
        target_labels = annotations["labels"].to(device)
        target_bboxes = annotations["boxes"].to(device)

        # Forward pass
        pred_class_logits, pred_bboxes = model(images)

        # Compute loss
        loss = loss_fn(pred_class_logits, pred_bboxes, target_labels, target_bboxes)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Loss: {loss.item()}")


# Example Usage
if __name__ == "__main__":
    # Example data
    dummy_images = torch.randn(100, 3, 224, 224)  # 10 images
    dummy_annotations = [
        {"boxes": torch.rand(5, 4), "labels": torch.randint(0, 91, (5,))} for _ in range(100)
    ]  # 5 objects per image

    dataset = ObjectDetectionDataset(dummy_images, dummy_annotations)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

    model = VisionTransformerForDetection(num_classes=91, num_queries=100).to("cuda")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_model(model, dataloader, optimizer, device="cuda")


Loss: 4.535745143890381
Loss: 4.537687301635742
Loss: 4.528448581695557
Loss: 4.5328288078308105
Loss: 4.525516986846924
Loss: 4.531887054443359
Loss: 4.530241966247559
Loss: 4.523733139038086
Loss: 4.527505397796631
Loss: 4.525564193725586
Loss: 4.523930549621582
Loss: 4.518754959106445
Loss: 4.519349098205566
Loss: 4.516720771789551
Loss: 4.5154523849487305
Loss: 4.5167012214660645
Loss: 4.5059733390808105
Loss: 4.504026412963867
Loss: 4.505673408508301
Loss: 4.502857685089111
Loss: 4.503025054931641
Loss: 4.503170490264893
Loss: 4.503547191619873
Loss: 4.500184059143066
Loss: 4.4996209144592285
Loss: 4.49416446685791
Loss: 4.495550155639648
Loss: 4.499051570892334
Loss: 4.482839107513428
Loss: 4.487951278686523
Loss: 4.484817981719971
Loss: 4.482458591461182
Loss: 4.483126640319824
Loss: 4.478682518005371
Loss: 4.478766441345215
Loss: 4.472090244293213
Loss: 4.466831684112549
Loss: 4.474144458770752
Loss: 4.465803146362305
Loss: 4.469417572021484
Loss: 4.460072040557861
Loss: 4.4606